# 1013 28일차

## 1. 가중치 확인

모델의 가중치를 꺼내 보는 속성 (`keras71_trainable_weights1`)

학습은 가중치 값을 바꾸는 과정이라, 학습 전후 값을 비교하거나 직접 계산할 때 꺼내 봄

1. `model.weights` : 모델의 모든 가중치. `Dense` 층마다 `kernel`(w)·`bias`(b) 두 개
2. `model.trainable_weights` : 학습할 때 바뀌는 가중치만. 기본은 `weights`와 같음

- `kernel` 모양은 `(입력 노드 수, 출력 노드 수)`, `bias`는 `(출력 노드 수,)`
- `kernel`은 랜덤 값, `bias`는 0으로 시작함
- 학습하면 `kernel`과 `bias`가 모두 바뀜

### 1-1. 직접 계산

꺼낸 가중치로 예측값을 손으로 계산 (`keras71_trainable_weights3`)

출력 노드 하나는 입력마다 `kernel`에서 그 노드 열의 값을 곱해 더하고, `bias`를 더한 값

```
출력 = 입력1 × w1 + 입력2 × w2 + ... + b
```

- `kernel`의 행은 들어오는 노드, 열은 나가는 노드
- 활성화 함수가 없으면 층마다 곱하고 더하기만 함. `relu`가 있으면 그 뒤에 음수를 0으로 바꿈
- 계산값이 `predict` 결과와 같으면 계산이 맞음

#### cf) 랜덤 고정

`seed`를 정해 실행할 때마다 같은 난수가 나오게 함

1. `random.seed(n)` : 파이썬 `random` 모듈
2. `np.random.seed(n)` : 넘파이
3. `tf.random.set_seed(n)` : 텐서플로. 가중치 초기값이 여기서 정해짐

- 같은 `seed`, 같은 층 구성이면 다른 파일에서도 처음 가중치가 같음

## 2. 동결

`trainable = False`로 가중치를 학습에서 빼는 설정 (`keras71_trainable_weights1`·`2`)

학습된 모델을 가져와 쓸 때, 내 데이터로 학습하는 동안 이미 학습된 가중치가 바뀌지 않게 함

1. `model.trainable = False` : 모델 전체를 동결함

- `weights` 개수는 그대로고 `trainable_weights`만 0개가 됨
- `summary()`에서 Trainable params 수가 Non-trainable params로 옮겨감
- 가중치 값은 남아 있음. `fit`을 해도 바뀌지 않아 예측은 처음 가중치로 낸 값 그대로
- `compile` 전에 설정함

## 3. VGG16

ImageNet으로 학습된 이미지 분류 CNN (`keras72_1_vgg16_include_top`)

큰 모델을 처음부터 학습시키려면 데이터와 시간이 많이 듦. 학습된 가중치째 가져와 씀

1. 구조 : 3×3 `Conv2D` 2~3개와 `MaxPooling2D`로 된 블록 5개 뒤에 `Flatten`, `Dense` 3개. 가중치가 있는 층이 16개
2. `weights='imagenet'` : ImageNet으로 학습된 가중치를 받음. `None`이면 구조만 가져오고 가중치는 랜덤
3. 출력 : ImageNet 1000개 클래스

- `from keras.applications import VGG16`
- 블록을 지날 때마다 가로·세로는 반, 필터 수는 64 → 128 → 256 → 512 → 512
- 파라미터 대부분이 마지막 `Dense` 층에 있음

### 3-1. include_top

모델 맨 뒤의 분류 부분(`Flatten`부터 출력 `Dense`까지)을 같이 가져올지 정하는 옵션

1. `True`(기본) : 분류 부분까지 가져옴. 출력은 1000개 클래스
2. `False` : Conv 블록까지만 가져옴. 출력은 특징 맵 `(가로, 세로, 512)`

- 클래스 수가 1000개가 아니면 `False`로 빼고 클래스 수에 맞는 `Dense`를 붙임
- `False`면 `input_shape`를 바꿀 수 있음. 가로·세로 32 이상

#### cf) top을 두고 input_shape 바꾸기

`include_top=True`, `weights='imagenet'`이면 `input_shape`는 `(224, 224, 3)`만 됨. 다른 값은 ValueError

- `Dense`는 받는 입력 수가 가중치 모양으로 정해져 있음
- 입력 크기가 바뀌면 `Flatten` 뒤 개수가 달라져 학습된 `Dense` 가중치를 쓸 수 없음
- `weights=None`이면 가중치를 새로 만들어서 다른 크기도 됨